# Similitud de conjuntos

Compara MinHash+LSH, SimHash y Jaccard exacto. Ejecuta todas las celdas. Este notebook contiene su código y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import hashlib
from collections import defaultdict
def stable_hash(value: object, salt: int = 0) -> int:
    return int.from_bytes(hashlib.blake2b(f"{salt}|{value}".encode(), digest_size=8).digest(), "little")


class BloomFilter:
    def __init__(self, capacity: int, error_rate: float = 0.01):
        self.bits_count = max(8, int(-capacity * math.log(error_rate) / math.log(2) ** 2))
        self.hashes = max(1, round(self.bits_count / max(capacity, 1) * math.log(2)))
        self.bits = bytearray((self.bits_count + 7) // 8)

    def _locations(self, value: object):
        for salt in range(self.hashes):
            yield stable_hash(value, salt) % self.bits_count

    def add(self, value: object) -> None:
        for pos in self._locations(value):
            self.bits[pos // 8] |= 1 << (pos % 8)

    def contains(self, value: object) -> bool:
        return all(self.bits[pos // 8] & (1 << (pos % 8)) for pos in self._locations(value))


class CountMinSketch:
    def __init__(self, width: int = 2048, depth: int = 5):
        self.width, self.depth = width, depth
        self.table = np.zeros((depth, width), dtype=np.uint32)

    def add(self, value: object, count: int = 1) -> None:
        for row in range(self.depth):
            self.table[row, stable_hash(value, row) % self.width] += count

    def estimate(self, value: object) -> int:
        return int(min(self.table[row, stable_hash(value, row) % self.width] for row in range(self.depth)))


class MinHashLSH:
    def __init__(self, permutations: int = 32, bands: int = 8):
        self.permutations, self.bands = permutations, bands
        self.rows = permutations // bands
        self.signatures: dict[int, tuple[int, ...]] = {}
        self.buckets: list[dict[tuple[int, ...], set[int]]] = [defaultdict(set) for _ in range(bands)]

    def signature(self, values: set[int]) -> tuple[int, ...]:
        return tuple(min((stable_hash(v, seed) for v in values), default=0) for seed in range(self.permutations))

    def add(self, key: int, values: set[int]) -> None:
        sig = self.signature(values)
        self.signatures[key] = sig
        for band in range(self.bands):
            start = band * self.rows
            self.buckets[band][sig[start:start + self.rows]].add(key)

    def candidates(self, key: int) -> set[int]:
        sig = self.signatures[key]
        hits: set[int] = set()
        for band in range(self.bands):
            start = band * self.rows
            hits.update(self.buckets[band][sig[start:start + self.rows]])
        hits.discard(key)
        return hits


def simhash(values: set[int], bits: int = 32) -> int:
    accumulator = [0] * bits
    for value in values:
        hashed = stable_hash(value)
        for bit in range(bits):
            accumulator[bit] += 1 if hashed & (1 << bit) else -1
    return sum((1 << bit) for bit, score in enumerate(accumulator) if score >= 0)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")

def relevant_sets(frame, maximum=250):
    groups = defaultdict(set)
    for product, user in zip(frame["product_id"], frame["user_id"]):
        groups[int(product)].add(int(user))
    return dict(sorted(groups.items(), key=lambda item: len(item[1]), reverse=True)[:maximum])


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


def run(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    data, source = load_events(data, max(sizes(mode)), seed)
    for n in sizes(mode):
        sets, reps = relevant_sets(data.iloc[:n]), repetitions(n, mode)
        keys = list(sets)
        if len(keys) < 2:
            continue
        query = keys[0]
        truth = {key for key in keys[1:] if jaccard(sets[query], sets[key]) >= 0.03}
        lsh = MinHashLSH()
        time_lsh, std_lsh, mem_lsh = measure(lambda: [lsh.add(key, values) for key, values in sets.items()], reps)
        for key, values in sets.items():
            lsh.add(key, values)
        candidates = lsh.candidates(query)
        recall = len(candidates & truth) / len(truth) if truth else 1.0
        record(rows, experiment="conjuntos", algorithm="MinHash+LSH", n=n, metric="recall", value=recall,
               time_ms=time_lsh, std_ms=std_lsh, memory_mb=mem_lsh, reps=reps, seed=seed, source=source)
        signatures = {key: simhash(values) for key, values in sets.items()}
        time_sh, std_sh, mem_sh = measure(lambda: {key: simhash(values) for key, values in sets.items()}, reps)
        limit = 10
        sh_candidates = {key for key in keys[1:] if hamming(signatures[query], signatures[key]) <= limit}
        sh_recall = len(sh_candidates & truth) / len(truth) if truth else 1.0
        record(rows, experiment="conjuntos", algorithm="SimHash", n=n, metric="recall", value=sh_recall,
               time_ms=time_sh, std_ms=std_sh, memory_mb=mem_sh, reps=reps, seed=seed, source=source)
        time_exact, std_exact, mem_exact = measure(
            lambda: [jaccard(sets[query], sets[key]) for key in keys[1:]], reps)
        record(rows, experiment="conjuntos", algorithm="Jaccard exacto", n=n, metric="recall", value=1.0,
               time_ms=time_exact, std_ms=std_exact, memory_mb=mem_exact, reps=reps, seed=seed, source=source)
    path = save(rows, "similitud", out)
    result = pd.read_csv(path)
    line_chart(result, "Similitud de conjuntos: tiempo de construcción", "similitud_tiempo.png", out)
    line_chart(result, "Similitud de conjuntos: recall", "similitud_recall.png", out, value="valor", log=False)
    return result

resultado = run(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED)
resultado


## Lectura

Jaccard exacto es la referencia de calidad. MinHash+LSH es la opción cuando el catálogo crece y hay que recuperar candidatos sin comparar todos los pares. SimHash queda como alternativa si su recall no alcanza el de los conjuntos de usuarios.
